# FREUID S0 — baseline_v0 training (IDE -> Colab GPU remote kernel)

S0 baseline: **tf_efficientnetv2_s.in21k** 384 px, recapture augmentation on train, recapture probe evaluated every epoch, checkpoint selected on **lowest probe_AuDET**.

Same Colab/ngrok setup as `run_baseline.ipynb` (Steps 1 and 3 are identical). Only STEP 2 cells differ.

> **Time:** 15 epochs x ~69k images on a T4 takes ~3-4 h. Use the reduced-epoch Colab variant (STEP 2, first cell) for a first complete run.

---

## STEP 1 — on Colab (browser; bring up the tunnel)

New Colab notebook -> Runtime type **GPU (T4)** -> add Secrets `NGROK_AUTH_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`. Paste the cells below into Colab and run them in order.

```python
# [Colab 1] check GPU + clone + install
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU -> set Runtime to GPU (T4)')
!pip install -q pyngrok
!git clone --depth 1 https://github.com/hyunsoochung-portfolio/kaggle_freuid_2026.git
%cd kaggle_freuid_2026
!pip install -q -e .   # reuse Colab's built-in torch (CUDA); installs only timm/kaggle/etc.
```

```python
# [Colab 2] Kaggle auth
import os, json
from google.colab import userdata
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': userdata.get('KAGGLE_USERNAME'), 'key': userdata.get('KAGGLE_KEY')}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json written')
```

```python
# [Colab 3] download data
!kaggle competitions download -c the-freuid-challenge-2026-ijcai-ecai -p data
!cd data && unzip -q -o '*.zip' && rm -f *.zip
!echo 'train imgs:' && ls data/train/train | wc -l && ls data
```

```python
# [Colab 4] (required) mount Google Drive -> checkpoints/, submissions/ auto-saved to Drive
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/freuid/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/freuid/submissions', exist_ok=True)
!rm -rf checkpoints submissions
!ln -s /content/drive/MyDrive/freuid/checkpoints checkpoints
!ln -s /content/drive/MyDrive/freuid/submissions submissions
print('checkpoints/, submissions/ -> linked to Drive')
```

```python
# [Colab 5] GPU kernel + ngrok tunnel -> print URL
import subprocess, time, socket
from google.colab import userdata
from pyngrok import conf, ngrok
conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN').strip()
JUPYTER_TOKEN = 'freuid'
for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
ngrok.kill()
log = open('/tmp/jl.log', 'w')
subprocess.Popen(['jupyter','server','--ip=0.0.0.0','--port=8888','--no-browser','--allow-root',
    f'--ServerApp.token={JUPYTER_TOKEN}','--ServerApp.password=','--ServerApp.allow_origin=*',
    '--ServerApp.disable_check_xsrf=True','--ServerApp.allow_remote_access=True',
    '--ServerApp.root_dir=/content/kaggle_freuid_2026'], stdout=log, stderr=subprocess.STDOUT)
for i in range(30):
    time.sleep(1)
    s = socket.socket()
    if s.connect_ex(('127.0.0.1', 8888)) == 0: s.close(); break
    s.close()
print(ngrok.connect(8888, 'http').public_url + '/?token=' + JUPYTER_TOKEN)
```

-> Copy the printed `https://....ngrok-free.app/?token=freuid`. (Keep this Colab tab open.)

---

## STEP 2 — in my IDE (this notebook)
1. Top-right **kernel picker** -> `Select Another Kernel...` -> `Existing Jupyter Server...` -> paste the URL -> `Python 3`
2. Run the cells below (all run on the Colab GPU)


In [ ]:
# Connection check: are we on the Colab GPU + is Drive linked?
import torch, os
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu (<- runtime is not GPU!)')
print('cwd:', os.getcwd())
print('checkpoints -> Drive?', os.path.realpath('checkpoints'))
!ls data && echo 'train imgs:' && ls data/train/train | wc -l


### Colab-friendly config — 5 epochs, all S0 features active

Writes `configs/baseline_v0_colab.yaml`: same backbone and all S0 keys (`augment: recapture`, `use_recapture_probe`, `checkpoint_metric: probe_audet`) but only **5 epochs** to finish in one Colab session.

Each epoch logs: `lr | train_loss | val_loss | AuDET | APCER@1%BPCER | probe_AuDET`. Checkpoint saved on **lowest probe_AuDET**.

To run the full 15 epochs instead, skip this cell and set `CONFIG` below to `configs/baseline_v0.yaml`.


In [ ]:
%%writefile configs/baseline_v0_colab.yaml
# S0 baseline, Colab variant: 5 epochs to finish in one session.
# All S0 features active: recapture aug, probe, probe_audet checkpointing.
name: baseline_v0_colab
seed: 42
data_dir: data
image_size: 384
val_fraction: 0.1
backbone: tf_efficientnetv2_s.in21k
pretrained: true
epochs: 5
batch_size: 32
lr: 3.0e-4
weight_decay: 1.0e-4
num_workers: 2

extra:
  augment: recapture
  use_recapture_probe: true
  recapture_probe_seed: 1234
  checkpoint_metric: probe_audet
  missing_id_score: 0.5


In [ ]:
# Train on full data (Colab GPU).
# Checkpoint -> checkpoints/baseline_v0_colab.pt -> auto-saved to Drive if linked.
# Sanity check (init BCE ~= ln2) runs automatically before epoch 1.
CONFIG = 'configs/baseline_v0_colab.yaml'   # or 'configs/baseline_v0.yaml' for 15 epochs
!python -m freuid.train --config {CONFIG}


In [ ]:
# Inference -> submission csv.
# backbone and image_size are always read from the checkpoint, not the config.
CKPT = 'checkpoints/baseline_v0_colab.pt'   # or 'checkpoints/baseline_v0.pt'
OUT  = 'submissions/baseline_v0_colab.csv'
!python -m freuid.infer --checkpoint {CKPT} --out {OUT}
# ^ prints integrity report: rows, unique scores, exact-zero count, min/max


### Getting the results out
If you ran STEP 1 [Colab 4] (Drive mount), `checkpoints/` and `submissions/` are **already auto-saved to `MyDrive/freuid/`**. In addition:
- Pull to your Mac: download from `MyDrive/freuid/submissions/` in Google Drive.
- **Submit to Kaggle directly** with the cell below (rules accepted + within daily limit):


In [ ]:
# (optional) Submit to Kaggle directly from Colab
OUT = 'submissions/baseline_v0_colab.csv'
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m 'S0 baseline_v0 recapture+probe'


## STEP 3 — (optional) auto-receive results into my local repo

STEP 1+2 save to Colab/Drive. To have `checkpoints/baseline_v0_colab.pt` appear in your local repo automatically:

> This is not a repo change — run once on your own machine.

**1. Install Google Drive desktop + sign in**
```bash
brew install --cask google-drive
# Open app -> sign in -> Settings -> prefer "Mirror files"
```

**2. Symlink the repo's `checkpoints/` to Drive**
```bash
DRIVE="$HOME/Library/CloudStorage/GoogleDrive-<email>/My Drive/freuid"
mkdir -p "$DRIVE/checkpoints"
cd <this-repo-path>
rmdir checkpoints 2>/dev/null
ln -s "$DRIVE/checkpoints" checkpoints
```

**Verify**
```bash
realpath checkpoints   # should point into CloudStorage/.../freuid/checkpoints
git status --short     # nothing checkpoints-related should appear
```

-> Colab training -> Drive -> Mac sync -> `checkpoints/baseline_v0_colab.pt` appears locally.

For submissions, use:
```bash
bash scripts/pull_submissions.sh
```
